<a href="https://colab.research.google.com/github/simodepth96/SERP-Clustering/blob/main/SERP_Clustering_DataForSEO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install umap-learn requests plotly openpyxl --quiet

import time
import json
import base64
import requests
import warnings

import pandas as pd
import numpy as np
import umap

from collections  import defaultdict
from collections import defaultdict, Counter

import plotly.express as px
import plotly.graph_objects as go

import scipy.cluster.hierarchy as sch
import networkx as nx

from itertools import combinations
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display


warnings.filterwarnings("ignore")
print("✅ Imports complete")

✅ Imports complete


In [ ]:
#@title Configuration

# DataForSEO credentials
DFS_LOGIN    = "login"
DFS_PASSWORD = "psw"

# Target location / language
# Location codes: https://docs.dataforseo.com/v3/appendix/google/locations/
LOCATION_CODE = 2826          # United Kingdom
LANGUAGE_CODE = "en"
SE_DOMAIN     = "google.co.uk"

#  How many organic results to keep per keyword
TOP_N_RESULTS = 10

#  Graph clustering hyperparameters
MIN_COMMON_URLS = 2
MIN_OVERLAP     = 0.3

print("✅ Configuration saved")

✅ Configuration saved. Now run Cell 3 to upload your keywords file.


In [ ]:
#@title Upload keywords from a .txt file

def load_keywords_from_txt(filepath: str) -> list[str]:
    """Parse a plain-text keyword file. One keyword per line."""
    keywords = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            kw = line.strip()
            if kw and not kw.startswith("#"):
                keywords.append(kw)
    return keywords


try:
    # Running in Google Colab
    from google.colab import files
    print("📂 A file picker will open. Select your keywords .txt file…")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]

    with open(filename, "wb") as f:
        f.write(uploaded[filename])
    KEYWORDS = load_keywords_from_txt(filename)

except ImportError:
    import tkinter as tk
    from tkinter import filedialog
    print("📂 Opening file dialog — select your keywords .txt file…")
    root = tk.Tk()
    root.withdraw()
    filepath = filedialog.askopenfilename(
        title="Select keywords .txt file",
        filetypes=[("Text files", "*.txt"), ("All files", "*.*")],
    )
    root.destroy()
    if not filepath:
        raise FileNotFoundError("No file selected. Please re-run this cell and choose a file.")
    KEYWORDS = load_keywords_from_txt(filepath)

# Validation
if not KEYWORDS:
    raise ValueError("❌ No keywords found in the file. Check the format (one keyword per line).")

# Deduplicate while preserving order
seen = set()
KEYWORDS = [kw for kw in KEYWORDS if not (kw.lower() in seen or seen.add(kw.lower()))]

print(f"\n✅ {len(KEYWORDS)} unique keyword(s) loaded:\n")
for i, kw in enumerate(KEYWORDS, 1):
    print(f"   {i:>3}. {kw}")

📂 A file picker will open. Select your keywords .txt file…


Saving keywords.txt to keywords (1).txt

✅ 46 unique keyword(s) loaded:

     1. trips to bruges christmas market
     2. weekend away ideas europe
     3. weekend away in february
     4. weekend break destinations
     5. weekend breaks abroad for couples
     6. weekend breaks in september
     7. weekend breaks to spain
     8. weekend in france
     9. weekend offers
    10. winter european city breaks
    11. amalfi coast weekend break
    12. american city breaks
    13. city breaks from belfast to london
    14. city breaks package deals
    15. city breaks sale
    16. city breaks suitable for families
    17. egypt city break
    18. estonia city breaks
    19. europe city breaks march
    20. europe weekend getaways
    21. european city break destinations
    22. european city breaks in january
    23. european weekend city breaks
    24. finland weekend breaks
    25. germany christmas market trips
    26. germany xmas market breaks
    27. golf weekends spain
    28. good

In [ ]:
#@title DataForSEO SERP scraper

DFS_AUTH = (DFS_LOGIN, DFS_PASSWORD)
HEADERS  = {"Content-Type": "application/json"}

def fetch_serp(keyword: str,
               location_code: int = LOCATION_CODE,
               language_code: str = LANGUAGE_CODE,
               se_domain: str     = SE_DOMAIN,
               top_n: int         = TOP_N_RESULTS,
               retries: int       = 3) -> list[dict]:
    endpoint = "https://api.dataforseo.com/v3/serp/google/organic/live/advanced"
    payload  = [
        {
            "keyword":                keyword,
            "location_code":          location_code,
            "language_code":          language_code,
            "se_domain":              se_domain,
            "depth":                  top_n,
            "device":                 "mobile",
            "os":                     "android",
            "calculate_rectangles":   True,
            "load_async_ai_overview": True,
        }
    ]
    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(
                endpoint,
                auth=DFS_AUTH,
                headers=HEADERS,
                data=json.dumps(payload),
                timeout=60,
            )

            if resp.status_code == 401:
                raise SystemExit("❌ 401 Unauthorized — fix DFS_LOGIN / DFS_PASSWORD and re-run.")

            resp.raise_for_status()
            data = resp.json()

            if data.get("status_code") != 20000:
                raise ValueError(f"API error {data.get('status_code')}: {data.get('status_message')}")

            tasks = data.get("tasks") or []
            if not tasks:
                raise ValueError("Response contained no tasks")

            task = tasks[0]
            if task.get("status_code") != 20000:
                raise ValueError(f"Task error {task.get('status_code')}: {task.get('status_message')}")

            result = task.get("result") or []
            if not result:
                raise ValueError("Task result is empty or None")

            rows          = []
            organic_count = 0

            for item in result[0].get("items") or []:
                item_type     = item.get("type")
                rank_group    = item.get("rank_group")
                rank_absolute = item.get("rank_absolute")
                rectangle_y   = (item.get("rectangle") or {}).get("y")
                url           = None

                if item_type == "organic":
                    if organic_count >= top_n:
                        continue
                    organic_count += 1
                    url = item.get("url")

                elif item_type == "ai_overview":
                    # First url from top-level references[] on the ai_overview item
                    for ref in item.get("references") or []:
                        if ref.get("type") == "ai_overview_reference" and ref.get("url"):
                            url = ref["url"]
                            break

                rows.append({
                    "keyword":       keyword,
                    "type":          item_type,
                    "url":           url,
                    "rank_group":    rank_group,
                    "rank_absolute": rank_absolute,
                    "rectangle_y":   rectangle_y,
                })

            return rows

        except SystemExit:
            raise
        except Exception as exc:
            print(f"  ⚠️  Attempt {attempt}/{retries} failed for '{keyword}': {exc}")
            if attempt < retries:
                time.sleep(2 ** attempt)

    print(f"  ❌  All retries exhausted for '{keyword}'. Returning empty list.")
    return []

print("✅ DataForSEO helper defined")

#  Fetch SERP results for all keywords
print(f"Fetching SERP data for {len(KEYWORDS)} keywords…\n")

keyword_urls: dict[str, list[str]] = {}
all_rows:     list[dict]           = []

pad = len(str(len(KEYWORDS)))

for i, kw in enumerate(KEYWORDS, 1):
    print(f"  [{i:>{pad}}/{len(KEYWORDS)}] {kw}")
    rows = fetch_serp(kw)

    organic_urls = [
        r["url"] for r in rows
        if r["type"] == "organic" and r["url"] and r["url"] != "None"
    ]
    keyword_urls[kw] = organic_urls
    all_rows.extend(rows)

    print(f"  {'':{pad}}        → {len(organic_urls)} organic  |  {len(rows)} total items")
    time.sleep(1)

print("\n✅ SERP fetch complete")

# Raw results DataFrame
raw_df = pd.DataFrame(all_rows, columns=[
    "keyword", "type", "url", "rank_group", "rank_absolute", "rectangle_y"
])

print(f"\nTotal SERP items : {len(raw_df):,}")
print(f"Unique keywords  : {raw_df['keyword'].nunique():,}")
print(f"SERP item types  : {sorted(raw_df['type'].dropna().unique().tolist())}")
print(f"Items with url   : {raw_df['url'].notna().sum():,}")
print(f"AI Overview rows : {(raw_df['type'] == 'ai_overview').sum():,}")
print(f"AI Overview w/url: {((raw_df['type'] == 'ai_overview') & raw_df['url'].notna()).sum():,}")
print(f"Items with rect_y: {raw_df['rectangle_y'].notna().sum():,}")
print()
display(raw_df.head(20))

✅ DataForSEO helper defined
Fetching SERP data for 46 keywords…

  [ 1/46] trips to bruges christmas market
            → 10 organic  |  13 total items
  [ 2/46] weekend away ideas europe
            → 10 organic  |  13 total items
  [ 3/46] weekend away in february
            → 9 organic  |  14 total items
  [ 4/46] weekend break destinations
            → 9 organic  |  13 total items
  [ 5/46] weekend breaks abroad for couples
            → 6 organic  |  10 total items
  [ 6/46] weekend breaks in september
            → 10 organic  |  14 total items
  [ 7/46] weekend breaks to spain
            → 10 organic  |  13 total items
  [ 8/46] weekend in france
            → 9 organic  |  17 total items
  [ 9/46] weekend offers
            → 10 organic  |  13 total items
  [10/46] winter european city breaks
            → 9 organic  |  15 total items
  [11/46] amalfi coast weekend break
            → 10 organic  |  14 total items
  [12/46] american city breaks
            → 9 organic  |  14

,keyword,type,url,rank_group,rank_absolute,rectangle_y
0,trips to bruges christmas market,organic,https://www.eurostar.com/uk-en/destinations/br...,1,1,156.0
1,trips to bruges christmas market,organic,https://www.tui.co.uk/holidays/christmas-marke...,2,2,333.0
2,trips to bruges christmas market,people_also_search,None,1,3,536.0
3,trips to bruges christmas market,organic,https://www.shearings.com/holidays/brussels-an...,3,4,971.0
4,trips to bruges christmas market,people_also_ask,None,1,5,1174.0
5,trips to bruges christmas market,organic,https://www.expedia.co.uk/Bruges-Christmas-Mar...,4,6,1475.0
6,trips to bruges christmas market,organic,https://www.milesmorgantravel.co.uk/blog/bruge...,5,7,1678.0
7,trips to bruges christmas market,organic,https://edwardscoaches.co.uk/tour/Lille-Bruges...,6,8,1855.0
8,trips to bruges christmas market,organic,https://www.leshuttle.com/uk-en/discover/trave...,7,9,2032.0
9,trips to bruges christmas market,organic,https://www.holidaypirates.com/others/christma...,8,10,2235.0


In [ ]:
#@title Clustering with TF-IDF similarity on scraped SERPs

def build_url_sets(keyword_urls: dict[str, list[str]]) -> dict[str, set]:
    return {kw: set(urls) for kw, urls in keyword_urls.items() if urls}

url_sets = build_url_sets(keyword_urls)
kw_list  = sorted(url_sets.keys())
n        = len(kw_list)
print(f"✅ URL sets built — {n} unique keywords")

#Overlap coefficient

def calculate_overlap_coefficient(set1: set, set2: set) -> float:
    """Overlap coefficient = |A ∩ B| / min(|A|, |B|)"""
    if not set1 or not set2:
        return 0.0
    return len(set1 & set2) / min(len(set1), len(set2))

# Graph-based clustering on shared URLs
def cluster_keywords(url_sets: dict,
                     min_common_urls: int   = MIN_COMMON_URLS,
                     min_overlap: float     = MIN_OVERLAP) -> list[set]:
    """
    Mirror of cluster_related_keywords() from standalone script.
    Builds a graph where edges = sufficient shared URLs + overlap,
    then returns connected components as clusters.
    """
    G = nx.Graph()
    G.add_nodes_from(url_sets.keys())

    for kw1, kw2 in combinations(url_sets.keys(), 2):
        common = url_sets[kw1] & url_sets[kw2]
        if len(common) >= min_common_urls:
            overlap = calculate_overlap_coefficient(url_sets[kw1], url_sets[kw2])
            if overlap >= min_overlap:
                G.add_edge(kw1, kw2, weight=overlap)

    return list(nx.connected_components(G))

clusters = cluster_keywords(url_sets)
print(f"✅ Graph clustering complete")
print(f"  Cluster count : {len(clusters)}")

# Map keyword to cluster id
kw_to_cluster = {}
for cid, cluster in enumerate(clusters):
    for kw in cluster:
        kw_to_cluster[kw] = cid

# Keywords with no edges end up as their own component — mark singletons as noise
singleton_ids = {cid for cid, c in enumerate(clusters) if len(c) == 1}

cluster_df = pd.DataFrame({
    "keyword": kw_list,
    "cluster": [kw_to_cluster.get(kw, -1) for kw in kw_list],
})
cluster_df["cluster"] = cluster_df.apply(
    lambda r: -1 if r["cluster"] in singleton_ids else r["cluster"], axis=1
)
cluster_df["cluster_label"] = cluster_df["cluster"].apply(
    lambda c: f"Cluster {c}" if c >= 0 else "Noise / Unclustered"
)

noise_count = (cluster_df["cluster"] == -1).sum()
print(f"  Noise points  : {noise_count}  (singletons / unconnected)")

# Shared-URL enrichment

def cluster_shared_urls(keywords: list[str], url_sets: dict) -> list[str]:
    """URLs shared by ≥2 keywords in the cluster."""
    if len(keywords) < 2:
        return []
    url_counts = defaultdict(int)
    for kw in keywords:
        for url in url_sets.get(kw, set()):
            url_counts[url] += 1
    return [url for url, cnt in sorted(url_counts.items(), key=lambda x: -x[1])
            if cnt >= 2]

# TF-IDF cluster

def identify_cluster_topic(keywords: list[str]) -> str:
    """
    Mirror of identify_cluster_topic() from standalone script.
    Uses TF-IDF to find the most central keyword.
    Falls back to shortest keyword if TF-IDF fails.
    """
    if len(keywords) == 1:
        return keywords[0]
    vectorizer = TfidfVectorizer(stop_words="english")
    try:
        tfidf_matrix = vectorizer.fit_transform(keywords)
        avg_similarities = []
        for i in range(len(keywords)):
            sims = [
                (tfidf_matrix[i] * tfidf_matrix[j].T).toarray()[0][0]
                for j in range(len(keywords)) if i != j
            ]
            avg_similarities.append(np.mean(sims) if sims else 0.0)
        return keywords[int(np.argmax(avg_similarities))]
    except Exception:
        return min(keywords, key=len)

# summary

cluster_ids     = sorted(set(cluster_df["cluster"]) - {-1})
cluster_summary = []

for cid in cluster_ids:
    kws    = cluster_df[cluster_df["cluster"] == cid]["keyword"].tolist()
    shared = cluster_shared_urls(kws, url_sets)

    overlaps = []
    for kw1, kw2 in combinations(kws, 2):
        overlaps.append(calculate_overlap_coefficient(
            url_sets.get(kw1, set()), url_sets.get(kw2, set())
        ))
    avg_overlap = round(np.mean(overlaps), 3) if overlaps else 0.0

    cluster_summary.append({
        "cluster_id":     cid,
        "cluster_name":   identify_cluster_topic(kws),
        "size":           len(kws),
        "keywords":       ", ".join(kws),
        "shared_urls":    len(shared),
        "avg_overlap":    avg_overlap,
        "top_shared_url": shared[0] if shared else "",
    })

# Noise group
noise_kws = cluster_df[cluster_df["cluster"] == -1]["keyword"].tolist()
if noise_kws:
    cluster_summary.append({
        "cluster_id":     -1,
        "cluster_name":   "Noise / Unclustered",
        "size":           len(noise_kws),
        "keywords":       ", ".join(noise_kws),
        "shared_urls":    0,
        "avg_overlap":    0.0,
        "top_shared_url": "",
    })

summary_df = pd.DataFrame(cluster_summary).sort_values("cluster_id").reset_index(drop=True)

print("✅ Cluster summary built\n")
display(summary_df)

✅ URL sets built — 46 unique keywords
✅ Graph clustering complete
  Cluster count : 23
  Noise points  : 17  (singletons / unconnected)
✅ Cluster summary built



,cluster_id,cluster_name,size,keywords,shared_urls,avg_overlap,top_shared_url
0,-1,Noise / Unclustered,17,"4 nights switzerland package, amalfi coast wee...",0,0.000,
1,1,european weekend city breaks,19,"best city break website, city breaks europe fe...",32,0.236,https://www.tui.co.uk/holidays/city-breaks
2,4,long weekend breaks spain,2,"long weekend breaks spain, weekend breaks to s...",7,0.778,https://www.easyjet.com/en/holidays/spain
3,9,city breaks suitable for families,2,"city breaks suitable for families, good city b...",7,0.778,https://www.loveholidays.com/holidays/city-bre...
4,13,germany christmas market trips,2,"germany christmas market trips, germany xmas m...",8,0.800,https://www.tui.co.uk/holidays/christmas-markets
5,15,italian city breaks,2,"italian city breaks, italy weekend trips",3,0.333,https://www.jet2holidays.com/city-breaks/itali...
6,21,turkey weekend break,2,"turkey weekend break, weekend breaks to turkey",8,0.800,https://www.onthebeach.co.uk/destinations/turkey


In [ ]:
#@title SERP Analysis

def analyze_above_fold(df, above_fold_threshold=400):
    df = df.copy()
    df['is_above_fold'] = df['rectangle_y'] < above_fold_threshold

    above_fold_count = df['is_above_fold'].sum()
    below_fold_count = len(df) - above_fold_count
    total = len(df)

    fig = go.Figure(data=[go.Bar(
        y=['Above Fold', 'Below Fold'],
        x=[above_fold_count, below_fold_count],
        text=[above_fold_count, below_fold_count],
        textposition='outside',
        textfont=dict(family='Trebuchet MS, sans-serif', size=14, color='rgba(50,50,50,0.8)'),
        orientation='h',
        marker=dict(
            color=['#FF8000', '#FFB366'],
            line=dict(color='rgba(0,0,0,0.3)', width=1.5)
        ),
        hovertemplate='<b>%{y}</b><br>Count: %{x}<br>Percentage: %{customdata:.2f}%<extra></extra>',
        customdata=[(above_fold_count / total * 100), (below_fold_count / total * 100)]
    )])

    fig.update_layout(
        title=dict(
            text='Distribution of Elements Above/Below Fold',
            y=0.95, x=0.5, xanchor='center', yanchor='top',
            font=dict(family='Raleway, sans-serif', size=24, color='rgba(0,0,0,0.85)')
        ),
        yaxis_title="", xaxis_title="",
        template="plotly_white",
        height=400, width=800,
        yaxis=dict(tickfont=dict(family='Roboto, sans-serif', size=16, color='rgba(0,0,0,0.75)'),
                   gridwidth=0.1, gridcolor='rgba(0,0,0,0.05)'),
        xaxis=dict(showgrid=True, gridwidth=0.1, gridcolor='rgba(0,0,0,0.05)',
                   zeroline=False, tickfont=dict(family='Roboto, sans-serif', size=14, color='rgba(0,0,0,0.6)')),
        hoverlabel=dict(bgcolor="white", font_size=16, font_family="Montserrat, sans-serif",
                        bordercolor="rgba(0,0,0,0.2)"),
        margin=dict(l=20, r=60, t=80, b=40),
        bargap=0.25,
        plot_bgcolor='rgba(250,250,250,0.95)'
    )
    fig.show()


    return above_fold_count, below_fold_count


def analyze_type_distribution(df):
    if 'type' not in df.columns:
        print("Column 'type' not found. Available columns:", df.columns.tolist())
        return

    type_counts = df['type'].value_counts().reset_index()
    type_counts.columns = ['Type', 'Count']
    type_counts['Percentage'] = (type_counts['Count'] / type_counts['Count'].sum() * 100).round(2)

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=type_counts['Type'],
        x=type_counts['Count'],
        text=type_counts['Count'],
        textposition='outside',
        textfont=dict(family='Trebuchet MS, sans-serif', size=14, color='rgba(50,50,50,0.8)'),
        orientation='h',
        marker=dict(
            color=type_counts['Count'],
            colorscale=px.colors.sequential.Plasma,
            line=dict(color='rgba(0,0,0,0.3)', width=1.5)
        ),
        hovertemplate='<b>%{y}</b><br>Count: %{x}<br>Percentage: %{customdata:.2f}%<extra></extra>',
        customdata=type_counts['Percentage']
    ))

    fig.update_layout(
        title=dict(
            text='Distribution of Result Types',
            y=0.97, x=0.5, xanchor='center', yanchor='top',
            font=dict(family='Raleway, sans-serif', size=28, color='rgba(0,0,0,0.85)')
        ),
        yaxis_title="", xaxis_title="",
        template="plotly_white",
        height=max(500, len(type_counts) * 45), width=950,
        yaxis=dict(categoryorder='total ascending',
                   tickfont=dict(family='Roboto, sans-serif', size=16, color='rgba(0,0,0,0.75)'),
                   gridwidth=0.1, gridcolor='rgba(0,0,0,0.05)', showgrid=False),
        xaxis=dict(showgrid=True, gridwidth=0.1, gridcolor='rgba(0,0,0,0.05)',
                   zeroline=False, tickfont=dict(family='Roboto, sans-serif', size=14, color='rgba(0,0,0,0.6)')),
        hoverlabel=dict(bgcolor="white", font_size=16, font_family="Montserrat, sans-serif",
                        bordercolor="rgba(0,0,0,0.2)"),
        margin=dict(l=20, r=60, t=80, b=40),
        bargap=0.25,
        plot_bgcolor='rgba(250,250,250,0.95)'
    )
    fig.show()

# Items like people_also_ask / related_searches often have no rectangle.
df_with_rect = raw_df[raw_df['rectangle_y'].notna()].copy()

analyze_type_distribution(raw_df)
analyze_above_fold(df_with_rect)

(np.int64(58), np.int64(526))

In [ ]:
#@title Cluster View & Export

bar_data = (
    summary_df[summary_df["cluster_id"] >= 0]
    .sort_values("size", ascending=True)
)

fig_bar = px.bar(
    bar_data,
    x="size",
    y="cluster_name",
    orientation="h",
    title="Number of Keywords per Cluster",
    labels={"size": "Keywords", "cluster_name": "Cluster"},
    template="plotly_white",
    color="size",
    color_continuous_scale="Blues",
    text="size",
)
fig_bar.update_traces(textposition="outside")
fig_bar.update_layout(showlegend=False, coloraxis_showscale=False)
fig_bar.show()


#Export results to CSV

output_csv  = "serp_clusters.csv"
output_raw  = "serp_raw_urls.csv"
output_sum  = "serp_cluster_summary.csv"

results_df.to_csv(output_csv, index=False)
raw_df.to_csv(output_raw, index=False)
summary_df.to_csv(output_sum, index=False)

try:
    from google.colab import files
    files.download(output_csv)
    files.download(output_sum)
    print("\n⬇️  Downloads triggered (check your browser)")
except ImportError:
    print("\n(Not running in Colab — files saved to current directory)")


In [ ]:
#@title SERP Domain Ranking Heatmap


def serp_heatmap(df, num_domains=10):
    """
    Heatmap of domain rankings across keywords.
    Expects columns: url (used to extract domain), keyword, rank_group.
    """
    df = df.copy()

    # Extract domain from URL if a 'domain' column isn't already present
    if 'domain' not in df.columns:
        df['domain'] = df['url'].dropna().apply(
            lambda u: u.split('/')[2].replace('www.', '') if isinstance(u, str) else None
        )

    df = df.rename(columns={'domain': 'displayLink', 'rank_group': 'rank'})
    df = df.dropna(subset=['displayLink', 'rank', 'keyword'])

    top_domains = df['displayLink'].value_counts()[:num_domains].index.tolist()
    top_df = df[df['displayLink'].isin(top_domains) & df['displayLink'].ne('')]

    top_df_counts_means = (
        top_df.groupby('displayLink', as_index=False)
        .agg({'rank': ['count', 'mean']})
    )
    top_df_counts_means.columns = ['displayLink', 'rank_count', 'rank_mean']
    top_df = (
        pd.merge(top_df, top_df_counts_means)
        .sort_values(['rank_count', 'rank_mean'], ascending=[False, True])
    )

    rank_counts = (
        top_df.groupby(['displayLink', 'rank'])
        .agg({'rank': ['count']})
        .reset_index()
    )
    rank_counts.columns = ['displayLink', 'rank', 'count']

    num_queries = df['keyword'].nunique()

    fig = go.Figure()
    fig.add_scatter(
        x=top_df['displayLink'].str.replace('www.', '', regex=True),
        y=top_df['rank'], mode='markers',
        marker={'size': 30, 'opacity': 1 / rank_counts['count'].max()}
    )
    fig.add_scatter(
        x=rank_counts['displayLink'].str.replace('www.', '', regex=True),
        y=rank_counts['rank'], mode='text',
        text=rank_counts['count']
    )

    for domain in rank_counts['displayLink'].unique():
        rc = rank_counts[rank_counts['displayLink'] == domain]
        label = domain.replace('www.', '')
        fig.add_scatter(x=[label], y=[0],  mode='text', text=str(rc['count'].sum()))
        fig.add_scatter(x=[label], y=[-1], mode='text',
                        text=format(rc['count'].sum() / num_queries, '.1%'))
        fig.add_scatter(x=[label], y=[-2], mode='text',
                        text=str(round(rc['rank'].mul(rc['count']).sum() / rc['count'].sum(), 2)))

    minrank = int(top_df['rank'].min())
    maxrank = int(top_df['rank'].max())
    fig.layout.yaxis.tickvals  = [-2, -1, 0] + list(range(minrank, maxrank + 1))
    fig.layout.yaxis.ticktext  = ['Avg. Pos.', 'Coverage', 'Total<br>appearances'] + list(range(minrank, maxrank + 1))
    fig.layout.height          = max(600, 100 + (maxrank - minrank) * 50)
    fig.layout.yaxis.title     = 'SERP Rank (number of appearances)'
    fig.layout.showlegend      = False
    fig.layout.paper_bgcolor   = '#eeeeee'
    fig.layout.plot_bgcolor    = '#eeeeee'
    fig.layout.autosize        = False
    fig.layout.margin.r        = 2
    fig.layout.margin.l        = 120
    fig.layout.margin.pad      = 0
    fig.layout.hovermode       = False
    fig.layout.yaxis.autorange = 'reversed'
    fig.layout.yaxis.zeroline  = False
    fig.layout.width           = 1100
    return fig


def analyze_domain_overlap(df, top_n_domains=15):
    """
    Calculate keyword overlap between the top N domains.
    Returns overlap_matrix (counts), overlap_percentages, common_keywords dict.
    """
    df = df.copy()
    if 'domain' not in df.columns:
        df['domain'] = df['url'].dropna().apply(
            lambda u: u.split('/')[2].replace('www.', '') if isinstance(u, str) else None
        )
    df = df.dropna(subset=['domain', 'keyword'])

    top_domains = df['domain'].value_counts().nlargest(top_n_domains).index.tolist()
    domain_keywords = {d: set(df[df['domain'] == d]['keyword']) for d in top_domains}

    overlap_matrix      = pd.DataFrame(0,   index=top_domains, columns=top_domains)
    overlap_percentages = pd.DataFrame(0.0, index=top_domains, columns=top_domains)
    common_keywords     = {}

    for dom1, dom2 in combinations(top_domains, 2):
        common = domain_keywords[dom1] & domain_keywords[dom2]
        common_keywords[(dom1, dom2)] = common
        n = len(common)
        overlap_matrix.loc[dom1, dom2] = n
        overlap_matrix.loc[dom2, dom1] = n
        overlap_percentages.loc[dom1, dom2] = n / len(domain_keywords[dom1]) * 100
        overlap_percentages.loc[dom2, dom1] = n / len(domain_keywords[dom2]) * 100

    return overlap_matrix, overlap_percentages, common_keywords


def create_overlap_heatmap(overlap_matrix, overlap_percentages):
    """
    Lower-triangle heatmap of domain keyword overlap, ordered by hierarchical clustering.
    """
    linkage = sch.linkage(overlap_matrix.values, method='average')
    idx     = sch.dendrogram(linkage, no_plot=True)['leaves']
    ordered_matrix      = overlap_matrix.iloc[idx, :].iloc[:, idx]
    ordered_percentages = overlap_percentages.iloc[idx, :].iloc[:, idx]
    ordered_domains     = ordered_matrix.columns

    mask             = np.tril(np.ones_like(ordered_matrix.values, dtype=bool))
    masked_matrix    = np.where(mask, ordered_matrix.values, np.nan)

    hover_text = np.empty(ordered_matrix.shape, dtype=object)
    for i in range(ordered_matrix.shape[0]):
        for j in range(ordered_matrix.shape[1]):
            if i >= j:
                hover_text[i, j] = (
                    f"<b>{ordered_domains[i]}</b> & <b>{ordered_domains[j]}</b><br>"
                    f"Overlapping keywords: <b>{ordered_matrix.iloc[i, j]}</b><br>"
                    f"Overlap %: <b>{ordered_percentages.iloc[i, j]:.1f}%</b>"
                )

    annotations = [
        dict(x=ordered_domains[j], y=ordered_domains[i],
             text=str(ordered_matrix.iloc[i, j]),
             showarrow=False,
             font=dict(color='black', size=11, family='Arial'),
             xanchor='center', yanchor='middle')
        for i in range(ordered_matrix.shape[0])
        for j in range(ordered_matrix.shape[1])
        if i >= j and ordered_matrix.iloc[i, j] > 10
    ]

    fig = go.Figure(data=go.Heatmap(
        z=masked_matrix,
        x=ordered_domains,
        y=ordered_domains,
        text=hover_text,
        hoverinfo='text',
        colorscale=[
            [0,   'white'],
            [0.2, '#FFE5CC'],
            [0.4, '#FFCC99'],
            [0.6, '#FFB366'],
            [0.8, '#FF9933'],
            [1,   '#FF8000'],
        ],
        colorbar=dict(
            title=dict(text='Number of<br>Overlapping Keywords', font=dict(size=12)),
            thickness=20, len=0.8, y=0.5
        )
    ))

    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)
    fig.update_layout(
        title=dict(
            text='<b>Domain Overlap Heatmap</b><br><span style="font-size:14px">Darker = more keyword overlap. Labels shown for overlaps > 10.</span>',
            x=0.5, y=0.92, xanchor='center', yanchor='top', font=dict(size=22)
        ),
        xaxis=dict(tickangle=45, side='top', tickfont=dict(size=12), zeroline=False, showline=False),
        yaxis=dict(tickangle=0, autorange='reversed', tickfont=dict(size=12), zeroline=False, showline=False),
        width=950, height=950,
        margin=dict(l=120, r=40, t=320, b=120),
        plot_bgcolor='white', paper_bgcolor='white',
        font=dict(size=13),
        annotations=annotations
    )
    return fig


#  Run all three analyses on raw_df
organic_df = raw_df[
    (raw_df['type'] == 'organic') & raw_df['url'].notna()
].copy()

print(f"Organic rows available for domain analysis: {len(organic_df):,}")
print(f"Unique domains (pre-filter): {organic_df['url'].apply(lambda u: u.split('/')[2].replace('www.','') if isinstance(u,str) else None).nunique():,}\n")

# Domain ranking heatmap
print("── Domain Ranking Heatmap ──────────────────────────────")
fig_heatmap = serp_heatmap(organic_df, num_domains=10)
fig_heatmap.show()

#  Domain overlap heatmap
print("── Domain Overlap Heatmap ──────────────────────────────")
overlap_matrix, overlap_percentages, common_keywords = analyze_domain_overlap(organic_df, top_n_domains=15)
fig_overlap = create_overlap_heatmap(overlap_matrix, overlap_percentages)
fig_overlap.show()

# Top overlapping domain pairs (console)
print("\n── Top 10 Domain Overlap Pairs ─────────────────────────")
for (dom1, dom2), kws in sorted(common_keywords.items(), key=lambda x: len(x[1]), reverse=True)[:10]:
    print(f"\n  {dom1}  ↔  {dom2}")
    print(f"  Common keywords : {len(kws)}")
    print(f"  Sample          : {', '.join(list(kws)[:5])}")

Organic rows available for domain analysis: 419
Unique domains (pre-filter): 128

── Domain Ranking Heatmap ──────────────────────────────


── Domain Overlap Heatmap ──────────────────────────────



── Top 10 Domain Overlap Pairs ─────────────────────────

  easyjet.com  ↔  tui.co.uk
  Common keywords : 25
  Sample          : europe weekend getaways, portugal weekend breaks, germany christmas market trips, prague xmas market breaks, european city break destinations

  easyjet.com  ↔  jet2holidays.com
  Common keywords : 23
  Sample          : portugal weekend breaks, weekend for two, prague xmas market breaks, europe city breaks march, european city breaks in january

  easyjet.com  ↔  lastminute.com
  Common keywords : 22
  Sample          : europe weekend getaways, portugal weekend breaks, 4 nights switzerland package, european weekend city breaks, european city break deals

  easyjet.com  ↔  onthebeach.co.uk
  Common keywords : 18
  Sample          : weekend break destinations, germany christmas market trips, weekend breaks abroad for couples, weekend offers, popular city breaks europe

  tui.co.uk  ↔  jet2holidays.com
  Common keywords : 18
  Sample          : weekend break d